# Dataset Preparation for MR-LFADS

This tutorial explains how to format your dataset to be compatible with the standard data structure for the MR-LFADS datamodule, `BasicDataModule`.

## 1. Data Structure

The `mrlfads.datamodules.BasicDataModule` is the standard datamodule for MR-LFADS. It expects an HDF5 file (`data.h5`) with the following structure:

```text
data.h5
├── session_index/
│   ├── area-<brain_area_name>      # neural activity (type: "hidden_state")
│   ├── inputs-<brain_area_name>    # inputs to a brain area (type: "inputs")
│   └── info                        # metadata (type: "info")
```

Each `session_index` group corresponds to a single recording session.

## 2. Dataset Requirements

Within each `session_index`, datasets must follow these conventions:

### (1) Neural Activity
- **Naming**: area-`<brain_area_name>`
- **Type**: `"hidden_state"`
- **Shape**: `(n_trials, n_time, n_neurons)`
- **Description**: Neural activity for a specific brain area `<brain_area_name>`
    
### (2) Inputs (optional)
- **Naming**: inputs-`<brain_area_name>`
- **Type**: `"inputs"`
- **Shape**: `(n_trials, n_time, n_input_features)`
- **Description**: External inputs associated with a specific brain area `<brain_area_name>`

### (3) Metadata (optional)
- **Naming**: arbitrary 
- **Type**: `"info"`  
- **Shape**: `(n_trials, n_time, n_info_dims)`  
- **Description**: Session-level metadata aligned to each trial and time step  

## 3. Test Your Dataset

To verify that your dataset loads correctly, specify the data file and brain area names:

In [1]:
filename = "binned_4_Chartreuse/subset_onesess"
area_names = ["MOp", "RSP","SSp-tr", "VISp"]

Then initialize and setup the datamodule:

In [2]:
import os
import sys 
import numpy as np
mrlfads_dir = "/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/MR-LFADS"
folder_path = os.path.abspath(mrlfads_dir)
sys.path.append(folder_path)
from mrlfads.datamodules import BasicDataModule

dm = BasicDataModule(
    filename=filename,
    area_names=area_names,
    session_idxs=['0'], # np.arange(19) if not single session
    train_val_split_seed=0,
)
dm.setup()

### Notes

* `filename` should point to your data.h5 file relative to config.paths.datapath
* `area_names` must match the suffixes used in your dataset (e.g., area-M1 → "M1")
* `session_idxs=[0]` loads the first session; adjust if multiple sessions are present

## 4. Under the Hood (Optional)

For a deeper understanding of how data is passed into the `mrlfads.model.MRLFADS` model, this section describes the internal structure.

### Batch Structure

For each `session_index`, data is passed to the model in the following format:

```text
session_idx
├── Main data (type: `mrlfads.utils.common_utils.Batch`)
│   ├── neural data
│   └── external inputs
└── Metadata (type: dict)
    ├── <metadata_name_1>
    └── ...
```

* Main data contains neural activity and inputs, and is stored as `model.current_batch`
* Metadata contains the trial-aligned metadata, and is stored as `model.current_info`
        
### Notes 

**Important Note** on `session_idx`: The session_idx used here refers to the index within the selected sessions, not the original dataset index. For example:

If your datamodule is initialized with session_idxs = [0, 4, 5], then batches will use:

`session_idx ∈ {0, 1, 2}`

corresponding to dataset sessions [0, 4, 5], respectively.

### Try It Yourself

If you have a stored MRLFADS model, you can inspect this structure directly using the following code:

In [3]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

checkpoint_path = "/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/training/MR-LFADS-results/01_memory_network_id2610232103"
config_path = os.path.join(checkpoint_path, 'configs', 'main.yaml')

In [4]:
from mrlfads.run import load

# Load the model and run one forward pass on the validation dataset
state_dict = load(
    config_path=config_path,
    validate=True,
)

# Extract the loaded model
model = state_dict['model']

/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/MR-LFADS/mrlfads/run.py:71: Hydra14MigrationWarning: 
version_base="1.1" selects Hydra 1.1 compatibility behavior.
Hydra 1.4 will remove this compatibility behavior.
Before upgrading to Hydra 1.4, set version_base="1.3" and test your application.
See https://hydra.cc/docs/upgrades/1.3_to_1.4/prepare_for_1_4/ for preparation instructions.
  with initialize_config_dir(version_base="1.1", config_dir=str(cfg_dir)):
Seed set to 42


Config path:  /Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/training/MR-LFADS-results/01_memory_network_id2610232103/configs/main.yaml
Checkpoint path:  /Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/training/MR-LFADS-results/01_memory_network_id2610232103


/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/MR-LFADS/mrlfads/run.py:178: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  ckpt = torch.load(ckpt_path, map_location=device)
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/envs/mrlfads/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/julietjiang/Library/Mobile Documents/com~apple~CloudDocs/Documents/UCHICAGO/BRENT/Lucas Collab/envs/mrlfads/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependenc

Validation DataLoader 0: 100%|██████████| 1/1 [00:41<00:00,  0.02it/s]


In [7]:
# Take one example session, one example area
session_idx = 0
area_name = model.area_names[0]

print(f'The shape of neural activity for area {area_name}, session {session_idx} is: ')
print(model.current_batch[session_idx].encod_data[area_name].shape)

The shape of neural activity for area A0, session 0 is: 
torch.Size([153, 200, 64])


In [8]:
print(f'The metadata keys are:')
print(model.current_info[session_idx].keys())

The metadata keys are:
dict_keys(['batch'])
